# PriceScope Stage 2: Exploratory Data Analysis

This notebook mirrors `eda/eda_report.py`. Use the script for reproducible headless runs and this notebook for interactive inspection.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

from eda.eda_report import (
    FIGURES_DIR,
    GPU_TIER_ORDER,
    comparable_retailer_delta,
    create_interactive_charts,
    create_static_charts,
    generate_insights,
    load_or_create_dataset,
    write_overview_summary,
)

sns.set_theme(style='whitegrid', palette='Set2')

## 1. Load Data

In [ ]:
df = load_or_create_dataset()
df.shape

In [ ]:
df.head()

## 2. Data Overview

In [ ]:
df.dtypes

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.select_dtypes('number').describe().T.round(2)

In [ ]:
display(df['brand'].value_counts())
display(df['retailer'].value_counts())

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['price_inr'], kde=True, bins=35, ax=axes[0])
axes[0].set_title('Laptop Price Distribution')
axes[0].set_xlabel('Price (INR)')
sns.boxplot(x=df['price_inr'], ax=axes[1])
axes[1].set_title('Laptop Price Boxplot')
axes[1].set_xlabel('Price (INR)')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.countplot(data=df, x='ram_tier', ax=axes[0])
axes[0].set_title('RAM Distribution')
sns.countplot(data=df, x='storage_tier', ax=axes[1])
axes[1].set_title('Storage Distribution')
axes[1].tick_params(axis='x', rotation=30)
sns.countplot(data=df, y='gpu_tier', order=GPU_TIER_ORDER, ax=axes[2])
axes[2].set_title('GPU Tier Distribution')
plt.tight_layout()

## 4. Bivariate and Relationship Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.boxplot(data=df, x='ram_tier', y='price_inr', ax=axes[0, 0])
axes[0, 0].set_title('Price vs RAM Tier')
sns.boxplot(data=df, x='storage_tier', y='price_inr', ax=axes[0, 1])
axes[0, 1].set_title('Price vs Storage Capacity')
axes[0, 1].tick_params(axis='x', rotation=30)
sns.boxplot(data=df, x='gpu_tier', y='price_inr', order=GPU_TIER_ORDER, ax=axes[1, 0])
axes[1, 0].set_title('Price vs GPU Tier')
axes[1, 0].tick_params(axis='x', rotation=25)
sns.boxplot(data=df, x='processor_brand', y='price_inr', ax=axes[1, 1])
axes[1, 1].set_title('Price vs Processor Brand')
plt.tight_layout()

In [ ]:
numeric_cols = ['processor_gen', 'ram_gb', 'storage_gb', 'gpu_vram_gb', 'display_size_inch', 'refresh_rate_hz', 'weight_kg', 'battery_wh', 'discount_pct', 'rating', 'rating_count', 'spec_score', 'price_inr']
plt.figure(figsize=(11, 8))
sns.heatmap(df[numeric_cols].corr(numeric_only=True), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlation Heatmap: Numeric Specs vs Price')
plt.tight_layout()

## 5. Retailer Comparison

In [ ]:
retailer_delta = comparable_retailer_delta(df)
retailer_delta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
if not retailer_delta.empty:
    sns.barplot(data=retailer_delta, x='retailer', y='mean_price_delta_pct', ax=axes[0])
    axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Comparable Spec Price Delta by Retailer')
axes[0].set_xlabel('Retailer')
axes[0].set_ylabel('Avg Price Delta (%)')
axes[0].tick_params(axis='x', rotation=25)
sns.boxplot(data=df, x='retailer', y='discount_pct', ax=axes[1])
axes[1].set_title('Discount % Distribution by Retailer')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()

## 6. Popular Configurations

In [ ]:
top_configs = df['configuration'].value_counts().head(12)
top_configs

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(x=top_configs.sort_values().values, y=top_configs.sort_values().index)
plt.title('Most Popular RAM + Storage + GPU Configurations')
plt.xlabel('Laptop Count')
plt.ylabel('Configuration')
plt.tight_layout()

## 7. Value-for-Money Exploratory Signal

In [ ]:
fig = px.scatter(
    df,
    x='spec_score',
    y='price_inr',
    color='brand',
    symbol='has_dedicated_gpu',
    hover_data=['model_name', 'retailer', 'ram_gb', 'storage_gb', 'gpu_model'],
    title='Interactive Price vs Naive Spec Score',
    labels={'spec_score': 'Naive Spec Score', 'price_inr': 'Price (INR)'},
)
fig.show()

## 8. Regenerate Reproducible Artifacts

In [ ]:
write_overview_summary(df)
static_paths = create_static_charts(df)
interactive_paths = create_interactive_charts(df)
insights = generate_insights(df)
print(f'Saved {len(static_paths)} static PNG charts to {FIGURES_DIR}')
print(f'Saved {len(interactive_paths)} Plotly PNG exports plus matching HTML files')
print(insights)